<a href="https://colab.research.google.com/github/Aditya-Raj-Kaushik/Vision-Transformer/blob/main/Vision_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json

path = "/content/Vision_Transformer.ipynb"

with open(path, "r", encoding="utf-8") as f:
    nb = json.load(f)

if "widgets" in nb.get("metadata", {}):
    del nb["metadata"]["widgets"]

with open(path, "w", encoding="utf-8") as f:
    json.dump(nb, f, indent=1)

print("Notebook fixed successfully.")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
import random

In [ ]:
torch.__version__

In [ ]:
torchvision.__version__

In [ ]:
!nvidia-smi

In [ ]:
torch.cuda.is_available()

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
print(f"Using Device: {device}")

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)
random.seed(42)

In [ ]:
BATCH_SIZE = 128
EPOCHS = 10
LEARNING_RATE = 3e-4
PATCH_SIZE = 4
NUM_CLASSES = 10
IMAGE_SIZE = 32
CHANNELS = 3
EMBED_DIM = 256
NUM_HEADS = 8
DEPTH = 6
MLP_DIM = 512
DROP_RATE = 0.1

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5), (0.5))
])

In [ ]:
transform_train = transforms.Compose([

    transforms.RandomCrop(32, padding=4),

    transforms.RandomHorizontalFlip(),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.2
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        (0.5, 0.5, 0.5),
        (0.5, 0.5, 0.5)
    )
])

In [ ]:
train_data = datasets.CIFAR10(
    root="data",
    train=True,
    download=True,
    transform=transform
)

In [ ]:
test_data = datasets.CIFAR10(
    root="data",
    train=False,
    download=True,
    transform=transform
)

In [ ]:
train_data

In [ ]:
len(train_data)

In [ ]:
len(test_data)

In [ ]:
train_loader = DataLoader(dataset=train_data, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(dataset=test_data, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
print(f"Dataloader: {train_loader, test_loader}")
print(f"Length of Trainloader: {len(train_loader)}")
print(f"Length of Testloader: {len(test_loader)}")

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, img_size, patch_size, in_channels, embed_dim):
        super().__init__()

        self.patch_size = patch_size

        self.proj = nn.Conv2d(
            in_channels,
            embed_dim,
            kernel_size=patch_size,
            stride=patch_size
        )

    def forward(self, x):
        x = self.proj(x)

        x = x.flatten(2)

        x = x.transpose(1, 2)

        return x


In [ ]:
class MLP(nn.Module):

    def __init__(self, in_features, hidden_features, out_features, drop_rate=0.1):

        super().__init__()

        self.fc1 = nn.Linear(
            in_features=in_features,
            out_features=hidden_features
        )

        self.fc2 = nn.Linear(
            in_features=hidden_features,
            out_features=out_features
        )

        self.drop = nn.Dropout(drop_rate)

    def forward(self, x):

        x = self.drop(F.gelu(self.fc1(x)))

        x = self.drop(self.fc2(x))

        return x

In [ ]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_dim, drop_rate):
        super().__init__()

        self.norm1 = nn.LayerNorm(embed_dim)

        self.attn = nn.MultiheadAttention(
            embed_dim,
            num_heads,
            dropout=drop_rate,
            batch_first=True
        )

        self.norm2 = nn.LayerNorm(embed_dim)

        self.mlp = MLP(
    in_features=embed_dim,
    hidden_features=mlp_dim,
    out_features=embed_dim,
    drop_rate=drop_rate
)

    def forward(self, x):
        x = x + self.attn(
            self.norm1(x),
            self.norm1(x),
            self.norm1(x)
        )[0]

        x = x + self.mlp(self.norm2(x))

        return x

In [ ]:
class VisionTransformer(nn.Module):
    def __init__(
        self,
        img_size,
        patch_size,
        in_channels,
        num_classes,
        embed_dim,
        depth,
        num_heads,
        mlp_dim,
        drop_rate
    ):
        super().__init__()

        self.patch_embed = PatchEmbedding(
            img_size,
            patch_size,
            in_channels,
            embed_dim
        )

        self.encoder = nn.Sequential(
            *[
                TransformerEncoderLayer(
                    embed_dim,
                    num_heads,
                    mlp_dim,
                    drop_rate
                )
                for _ in range(depth)
            ]
        )

        self.norm = nn.LayerNorm(embed_dim)

        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        x = self.patch_embed(x)

        x = self.encoder(x)

        x = self.norm(x)

        cls_token = x[:, 0]

        return self.head(cls_token)

In [ ]:
model = VisionTransformer(
    IMAGE_SIZE, PATCH_SIZE, CHANNELS, NUM_CLASSES, EMBED_DIM, DEPTH, NUM_HEADS, MLP_DIM, DROP_RATE
).to(device)

In [ ]:
model

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=LEARNING_RATE)

In [ ]:
criterion

In [ ]:
optimizer

In [ ]:
def train(model, loader, optimizer, criterion):
    model.train()

    total_loss, correct = 0, 0

    for x,y in loader:
      x,y = x.to(device), y.to(device)

      optimizer.zero_grad()

      y_pred = model(x)

      loss = criterion(y_pred, y)

      loss.backward()

      optimizer.step()

      total_loss += loss.item()*x.size(0)

      correct += (y_pred.argmax(1) == y).sum().item()

    return total_loss/len(loader.dataset), correct/len(loader.dataset)

In [ ]:
def evaluate(model, loader):
  model.eval()
  correct = 0
  with torch.inference_mode():
    for x,y in loader:
      x,y = x.to(device), y.to(device)
      y_pred = model(x)
      correct += (y_pred.argmax(dim=1) == y).sum().item()
  return correct/len(loader.dataset)

In [ ]:
from tqdm.auto import tqdm

In [ ]:
train_accuracies, test_accuracies = [], []

for epoch in tqdm(range(EPOCHS)):

    train_loss, train_acc = train(
        model,
        train_loader,
        optimizer,
        criterion
    )

    test_acc = evaluate(
        model,
        test_loader
    )

    train_accuracies.append(train_acc)
    test_accuracies.append(test_acc)

    print(
        f"Epoch: {epoch+1}/{EPOCHS}, "
        f"Train loss: {train_loss:.4f}, "
        f"Train acc: {train_acc:.4f}%, "
        f"Test acc: {test_acc:.4f}%"
    )

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch: 1/10, Train loss: 1.7298, Train acc: 0.3614%, Test acc: 0.4317%
Epoch: 2/10, Train loss: 1.4370, Train acc: 0.4784%, Test acc: 0.4937%
Epoch: 3/10, Train loss: 1.3089, Train acc: 0.5262%, Test acc: 0.5361%
Epoch: 4/10, Train loss: 1.2245, Train acc: 0.5609%, Test acc: 0.5540%
Epoch: 5/10, Train loss: 1.1540, Train acc: 0.5857%, Test acc: 0.5641%
Epoch: 6/10, Train loss: 1.1012, Train acc: 0.6028%, Test acc: 0.5360%
Epoch: 7/10, Train loss: 1.0515, Train acc: 0.6217%, Test acc: 0.5648%


In [ ]:
plt.plot(train_accuracies, label="Train Accuracy")
plt.plot(test_accuracies, label="Test Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("Accuracy vs Epochs")
plt.legend()
plt.show()

In [ ]:
def predict_and_plot_grid(
    model,
    dataset,
    classes,
    grid_size=3
):

    model.eval()

    fig, axes = plt.subplots(
        grid_size,
        grid_size,
        figsize=(9, 9)
    )

    for i in range(grid_size):

        for j in range(grid_size):

            idx = random.randint(0, len(dataset) - 1)

            img, true_label = dataset[idx]

            input_tensor = img.unsqueeze(dim=0).to(device)

            with torch.inference_mode():

                output = model(input_tensor)

                _, predicted = torch.max(output.data, 1)

            img = img / 2 + 0.5

            npimg = img.cpu().numpy()

            axes[i, j].imshow(
                np.transpose(npimg, (1, 2, 0))
            )

            truth = classes[true_label] == classes[predicted.item()]

            if truth:
              color = "g"
            else:
              color = "r"

            axes[i, j].set_title(
                f"True: {classes[true_label]} | Pred: {classes[predicted.item()]}",
                color=color
            )

            axes[i, j].axis("off")

    plt.tight_layout()

    plt.show()

In [ ]:
predict_and_plot_grid(
    model,
    test_data,
    classes=test_data.classes,
    grid_size=3
)
